# Paper Financial Comparison Plots

This notebook creates separate publication-quality financial figures from saved `daily_financial_detail.csv` files. Optimization notebooks only save CSV outputs; all paper figures are centralized here.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib import dates as mdates
from matplotlib.patches import Patch

PROJECT = Path.cwd()
OUT_DIR = PROJECT / "Paper_Figures" / "financial_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9.5,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SCENARIOS = [
    ("Shrinking", "Perfect", "Shrinking-Perfect", PROJECT / "Results_Shrinking/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Shrinking", "Persistence", "Shrinking-Persistence", PROJECT / "Results_Shrinking/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Rolling", "Perfect", "Rolling-Perfect", PROJECT / "Validation_Results/June_2025/formal_runs/gate_e_20260726_c8250f11/rolling/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Rolling", "Persistence", "Rolling-Persistence", PROJECT / "Validation_Results/June_2025/formal_runs/gate_e_20260726_c8250f11/rolling/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv"),
]
RETAIL_ONLY_SOURCES = {
    "Shrinking-Perfect": PROJECT / "Results_Shrinking/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv",
    "Shrinking-Persistence": PROJECT / "Results_Shrinking/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv",
    "Rolling-Perfect": PROJECT / "Results_Rolling/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv",
    "Rolling-Persistence": PROJECT / "Results_Rolling/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv",
}
SCENARIO_ORDER = [s[2] for s in SCENARIOS]
CASE_ORDER = ["Retail only", "Both"]
CASE_COLORS = {"Retail only": "#D55E00", "Both": "#0072B2"}
COMPONENT_COLORS = {
    "EV Revenue": "#009E73",
    "WM Revenue": "#0072B2",
    "TOU Cost": "#CC79A7",
    "PD Cost": "#E69F00",
    "NCD Cost": "#6A3D9A",
}
METRICS = ["Total Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost", "EV Revenue"]
DECOMP_GROUP_COLORS = {
    "WM EV": "#0072B2",
    "WM BESS": "#009E73",
    "NWM EV": "#E69F00",
    "NWM BESS": "#6A3D9A",
}
DECOMP_COMPONENTS = [
    "WM EV Energy", "WM EV Capacity",
    "WM BESS Energy", "WM BESS Capacity",
    "NWM EV Energy", "NWM EV Capacity",
    "NWM BESS Energy", "NWM BESS Capacity",
]

def dollars(x, pos=None):
    sign = "-" if x < 0 else ""
    x = abs(x)
    if x >= 1000:
        return f"{sign}${x/1000:.0f}k"
    return f"{sign}${x:.0f}"

def save_fig(fig, filename):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(path.relative_to(PROJECT))
    return path


In [2]:
frames = []
missing = []
for mpc, forecast, scenario, path in SCENARIOS:
    if not path.exists():
        missing.append(path.relative_to(PROJECT))
        continue
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df[df["Case"] == "Both"].copy()
    both_dates = set(df["Date"])
    retail_path = RETAIL_ONLY_SOURCES[scenario]
    if retail_path.exists():
        retail_df = pd.read_csv(retail_path)
        retail_df["Date"] = pd.to_datetime(retail_df["Date"])
        retail_df = retail_df[(retail_df["Case"] == "Retail only") & retail_df["Date"].isin(both_dates)].copy()
        retail_counts = retail_df.groupby("Date").size()
        retail_complete = bool(both_dates) and set(retail_counts.index) == both_dates and bool((retail_counts == 1).all())
        retail_fresh = retail_path.stat().st_mtime >= path.stat().st_mtime
        if retail_complete and retail_fresh:
            df = pd.concat([df, retail_df], ignore_index=True, sort=False)
        elif retail_complete and not retail_fresh:
            print(f"Ignoring stale Retail-only source for {scenario}; rerun that Retail-only case before plotting.")
        elif not retail_df.empty:
            print(f"Ignoring incomplete Retail-only rows for {scenario}: {retail_df['Date'].nunique()} of {len(both_dates)} formal dates.")
    df["MPC"] = mpc
    df["Forecast"] = forecast
    df["Scenario"] = scenario
    frames.append(df)

if missing:
    print("Missing input files:")
    for item in missing:
        print(" -", item)
if not frames:
    raise FileNotFoundError("No daily_financial_detail.csv files found.")

all_df = pd.concat(frames, ignore_index=True)
all_df = all_df[all_df["Case"].isin(CASE_ORDER)].copy()
for col in METRICS + ["WM BESS Total", "WM EV Total", "WM Energy Revenue", "WM Capacity Revenue"] + DECOMP_COMPONENTS:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors="coerce")

summary = all_df.groupby(["Scenario", "MPC", "Forecast", "Case"], as_index=False)[METRICS].sum(numeric_only=True)

wide = all_df.pivot_table(index=["Scenario", "MPC", "Forecast", "Date"], columns="Case", values=METRICS, aggfunc="sum")
delta_df = pd.DataFrame(index=wide.index).reset_index()
for metric in METRICS:
    if (metric, "Both") in wide.columns and (metric, "Retail only") in wide.columns:
        delta_df[metric] = (wide[(metric, "Both")] - wide[(metric, "Retail only")]).values

print(f"Loaded {len(all_df)} case-day rows from {len(frames)} scenario files.")
print(f"Date range: {all_df['Date'].min().date()} to {all_df['Date'].max().date()}")
summary

Loaded 240 case-day rows from 4 scenario files.
Date range: 2025-06-01 to 2025-06-30


,Scenario,MPC,Forecast,Case,Total Revenue,WM Revenue,TOU Cost,PD Cost,NCD Cost,EV Revenue
0,Rolling-Perfect,Rolling,Perfect,Both,10733.15,4347.03,-12300.80,-279.25,-4427.75,23393.92
1,Rolling-Perfect,Rolling,Perfect,Retail only,6240.92,0.00,-12414.51,-310.73,-4427.74,23393.92
2,Rolling-Persistence,Rolling,Persistence,Both,10485.28,4421.87,-11092.27,-467.84,-5770.41,23393.92
3,Rolling-Persistence,Rolling,Persistence,Retail only,6549.50,0.00,-11631.63,-275.30,-4937.48,23393.92
4,Shrinking-Perfect,Shrinking,Perfect,Both,10552.67,4310.96,-12413.73,-310.73,-4427.75,23393.92
5,Shrinking-Perfect,Shrinking,Perfect,Retail only,6240.92,0.00,-12414.51,-310.73,-4427.74,23393.92
6,Shrinking-Persistence,Shrinking,Persistence,Both,10985.42,4424.62,-11527.35,-527.59,-4778.15,23393.92
7,Shrinking-Persistence,Shrinking,Persistence,Retail only,5356.72,0.00,-11611.26,-492.94,-5932.99,23393.92


## Period-aware paper figures

Generate combined and per-month financial figures from the currently saved dispatch results.


In [3]:
# Period-aware financial analysis for the current saved run.
# This cell uses the already-loaded all_df and regenerates paper-quality figures for:
#   1) the full available date range, and
#   2) each calendar month present in the result CSVs.

PERIOD_ROOT = OUT_DIR
PERIOD_ROOT.mkdir(parents=True, exist_ok=True)


def _period_label(df):
    start = pd.Timestamp(df["Date"].min()).strftime("%Y-%m-%d")
    end = pd.Timestamp(df["Date"].max()).strftime("%Y-%m-%d")
    return f"{start}_to_{end}"


def _make_period_summary(df, out_dir):
    period_summary = df.groupby(["Scenario", "MPC", "Forecast", "Case"], as_index=False)[METRICS].sum(numeric_only=True)
    period_summary.to_csv(out_dir / "financial_summary_by_scenario_case.csv", index=False)

    wide = df.pivot_table(index=["Scenario", "MPC", "Forecast", "Date"], columns="Case", values=METRICS, aggfunc="sum")
    period_delta = pd.DataFrame(index=wide.index).reset_index()
    for metric in METRICS:
        if (metric, "Both") in wide.columns and (metric, "Retail only") in wide.columns:
            period_delta[metric] = (wide[(metric, "Both")] - wide[(metric, "Retail only")]).values
    period_delta.to_csv(out_dir / "financial_delta_both_minus_retail_daily.csv", index=False)
    return period_summary, period_delta


def _save_period_fig(fig, out_dir, filename):
    path = out_dir / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(path.relative_to(PROJECT))
    return path


def plot_period_financials(period_df, label, title_suffix):
    out_dir = PERIOD_ROOT / label
    out_dir.mkdir(parents=True, exist_ok=True)
    period_df = period_df.copy()
    available_cases = set(period_df['Case'].dropna())
    paired_cases_complete = True
    for scenario in SCENARIO_ORDER:
        scenario_df = period_df[period_df['Scenario'] == scenario]
        per_date_cases = scenario_df.groupby('Date')['Case'].agg(set)
        if scenario_df.empty or per_date_cases.empty or not per_date_cases.map(lambda cases: set(CASE_ORDER).issubset(cases)).all():
            paired_cases_complete = False
            break
    if not paired_cases_complete:
        period_summary = period_df.groupby(['Scenario', 'MPC', 'Forecast', 'Case'], as_index=False)[METRICS].sum(numeric_only=True)
        period_summary.to_csv(out_dir / 'financial_summary_by_scenario_case.csv', index=False)
        stale_delta = out_dir / 'financial_delta_both_minus_retail_daily.csv'
        if stale_delta.exists():
            stale_delta.unlink()
        stale_paired_figures = [
            'fig_financial_violin_daily_total.png',
            'fig_financial_delta_heatmap.png',
            'fig_financial_case_comparison_subplots.png',
            'fig_financial_case_value_stack_subplots.png',
            'fig_profit_decomposition_daily.png',
            'fig_profit_decomposition_period_total.png',
        ]
        for stale_name in stale_paired_figures:
            stale_path = out_dir / stale_name
            if stale_path.exists():
                stale_path.unlink()
        if all(col in period_df.columns for col in DECOMP_COMPONENTS):
            both_decomp = period_df[period_df['Case'] == 'Both'][['Scenario', 'Date', 'Total Revenue'] + DECOMP_COMPONENTS].copy()
            both_decomp = both_decomp.sort_values(['Scenario', 'Date'])
            both_decomp['Decomposition Residual'] = both_decomp['Total Revenue'] - both_decomp[DECOMP_COMPONENTS].sum(axis=1)
            both_decomp.to_csv(out_dir / 'profit_decomposition_daily.csv', index=False)
            decomp_total = both_decomp.groupby('Scenario', as_index=False)[['Total Revenue'] + DECOMP_COMPONENTS + ['Decomposition Residual']].sum(numeric_only=True)
            decomp_total = decomp_total.set_index('Scenario').reindex(SCENARIO_ORDER).dropna(how='all').reset_index()
            decomp_total.to_csv(out_dir / 'profit_decomposition_period_total.csv', index=False)
        print(f"Updated available-case CSVs for {label} using {sorted(available_cases)}. Skipping only Retail-only/Both delta figures; formal review figures are exported below to Paper_Figures.")
        return period_summary, pd.DataFrame()
    period_summary, period_delta = _make_period_summary(period_df, out_dir)

    # 1) Daily total value violin: distribution over the selected period.
    fig, ax = plt.subplots(figsize=(7.6, 4.2), constrained_layout=True)
    positions, data, colors, labels = [], [], [], []
    x = 1.0
    for scenario in SCENARIO_ORDER:
        if scenario not in set(period_df["Scenario"]):
            continue
        for case in CASE_ORDER:
            vals = period_df[(period_df["Scenario"] == scenario) & (period_df["Case"] == case)]["Total Revenue"].dropna().values
            if len(vals) == 0:
                continue
            positions.append(x)
            data.append(vals)
            colors.append(CASE_COLORS[case])
            labels.append(scenario.replace("-", "\n") + "\n" + case)
            x += 0.82
        x += 0.45
    if data:
        parts = ax.violinplot(data, positions=positions, widths=0.58, showmedians=True, showextrema=False)
        for body, color in zip(parts["bodies"], colors):
            body.set_facecolor(color)
            body.set_edgecolor("#222222")
            body.set_alpha(0.28)
            body.set_linewidth(0.8)
        parts["cmedians"].set_color("#222222")
        parts["cmedians"].set_linewidth(1.1)
        rng = np.random.default_rng(7)
        for pos, vals, color in zip(positions, data, colors):
            jitter = rng.normal(0, 0.055, size=len(vals))
            ax.scatter(np.full(len(vals), pos) + jitter, vals, s=13, color=color, edgecolor="white", linewidth=0.35, alpha=0.78, zorder=3)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylabel("Daily net financial value (US$)")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(axis="y", color="#bdbdbd", alpha=0.32, linewidth=0.7)
    ax.legend(handles=[Patch(facecolor=CASE_COLORS[c], edgecolor="none", label=c, alpha=0.75) for c in CASE_ORDER], frameon=False, loc="upper left")
    ax.set_title(f"Daily financial value distribution ({title_suffix})", fontweight="bold")
    _save_period_fig(fig, out_dir, "fig_financial_violin_daily_total.png")
    plt.close(fig)

    # 2) Heatmap: incremental Both minus Retail value by component.
    heat_metrics = ["Total Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost"]
    heat = period_delta.groupby("Scenario")[heat_metrics].sum(numeric_only=True).reindex(SCENARIO_ORDER)
    fig, ax = plt.subplots(figsize=(7.0, 3.35), constrained_layout=True)
    max_abs = float(np.nanmax(np.abs(heat.values))) if heat.size and np.isfinite(heat.values).any() else 1.0
    if max_abs == 0:
        max_abs = 1.0
    im = ax.imshow(heat.values, cmap="RdBu", vmin=-max_abs, vmax=max_abs, aspect="auto")
    ax.set_xticks(np.arange(len(heat_metrics)))
    ax.set_xticklabels([m.replace(" Revenue", "\nRevenue").replace(" Cost", "\nCost") for m in heat_metrics])
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels([s.replace("-", "\n") for s in heat.index])
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            val = heat.iloc[i, j]
            text = f"{val/1000:.1f}k" if abs(val) >= 1000 else f"{val:.0f}"
            ax.text(j, i, text, ha="center", va="center", fontsize=8, color="#111111")
    ax.set_title(f"Wholesale-enabled incremental value ({title_suffix})", fontweight="bold")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.025)
    cbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    cbar.set_label("Both minus retail-only (US$)")
    _save_period_fig(fig, out_dir, "fig_financial_delta_heatmap.png")
    plt.close(fig)

    # 3) Daily Retail-only vs Both comparison in separate scenario panels.
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 5.8), sharex=False, sharey=True, constrained_layout=True)
    axes = axes.flatten()
    bar_w = 0.38
    for ax, scenario in zip(axes, SCENARIO_ORDER):
        sub = period_df[period_df["Scenario"] == scenario].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        dates = sorted(sub["Date"].unique())
        x = np.arange(len(dates))
        retail = sub[sub["Case"] == "Retail only"].set_index("Date").reindex(dates)["Total Revenue"].values
        both = sub[sub["Case"] == "Both"].set_index("Date").reindex(dates)["Total Revenue"].values
        delta = both - retail
        ax.bar(x - bar_w/2, retail, width=bar_w, color=CASE_COLORS["Retail only"], alpha=0.78, label="Retail only")
        ax.bar(x + bar_w/2, both, width=bar_w, color=CASE_COLORS["Both"], alpha=0.78, label="Both")
        ax.plot(x, delta, color="#111111", marker="o", markersize=3.2, linewidth=1.0, label="Both - Retail")
        ax.axhline(0, color="#333333", linewidth=0.8)
        ax.set_title(scenario.replace("-", " + "), fontweight="bold", pad=5)
        tick_step = max(1, int(np.ceil(len(dates) / 10)))
        ax.set_xticks(x[::tick_step])
        ax.set_xticklabels([pd.Timestamp(d).strftime("%m/%d") for d in dates[::tick_step]], rotation=0)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
        ax.grid(axis="y", color="#bdbdbd", alpha=0.30, linewidth=0.7)
        if len(delta):
            worst_idx = np.argsort(delta)[:min(2, len(delta))]
            for idx in worst_idx:
                if delta[idx] < 0:
                    ax.annotate(f"{delta[idx]:.0f}", (x[idx], delta[idx]), textcoords="offset points", xytext=(0, -12), ha="center", fontsize=7.5, color="#111111")
    axes[0].set_ylabel("Daily value / delta (US$)")
    axes[2].set_ylabel("Daily value / delta (US$)")
    handles = [
        Patch(facecolor=CASE_COLORS["Retail only"], label="Retail only", alpha=0.78),
        Patch(facecolor=CASE_COLORS["Both"], label="Both", alpha=0.78),
        plt.Line2D([0], [0], color="#111111", marker="o", linewidth=1.0, label="Both - Retail"),
    ]
    fig.legend(handles=handles, ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.03))
    fig.suptitle(f"Retail-only and wholesale-enabled daily values ({title_suffix})", fontsize=12, fontweight="bold", y=1.08)
    _save_period_fig(fig, out_dir, "fig_financial_case_comparison_subplots.png")
    plt.close(fig)

    # 4) Value stack for Retail-only and Both in each scenario.
    components = ["EV Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost"]
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 5.8), sharey=True, constrained_layout=True)
    axes = axes.flatten()
    for ax, scenario in zip(axes, SCENARIO_ORDER):
        scen_sum = period_summary[period_summary["Scenario"] == scenario].set_index("Case").reindex(CASE_ORDER)
        xs = np.arange(len(CASE_ORDER))
        pos_bottom = np.zeros(len(CASE_ORDER))
        neg_bottom = np.zeros(len(CASE_ORDER))
        for comp in components:
            vals = scen_sum[comp].fillna(0).values
            bottoms = np.where(vals >= 0, pos_bottom, neg_bottom)
            ax.bar(xs, vals, bottom=bottoms, width=0.58, color=COMPONENT_COLORS[comp], edgecolor="white", linewidth=0.55, label=comp)
            pos_bottom += np.where(vals >= 0, vals, 0)
            neg_bottom += np.where(vals < 0, vals, 0)
        net = scen_sum["Total Revenue"].fillna(0).values
        ax.plot(xs, net, color="#111111", marker="o", markersize=4.0, linewidth=1.1)
        for xi, yi in zip(xs, net):
            ax.annotate(f"${yi:,.0f}", (xi, yi), textcoords="offset points", xytext=(0, 6 if yi >= 0 else -12), ha="center", fontsize=7.5)
        ax.axhline(0, color="#333333", linewidth=0.8)
        ax.set_title(scenario.replace("-", " + "), fontweight="bold", pad=5)
        ax.set_xticks(xs)
        ax.set_xticklabels(CASE_ORDER)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
        ax.grid(axis="y", color="#bdbdbd", alpha=0.30, linewidth=0.7)
    axes[0].set_ylabel("Period value (US$)")
    axes[2].set_ylabel("Period value (US$)")
    handles = [Patch(facecolor=COMPONENT_COLORS[c], edgecolor="none", label=c) for c in components]
    fig.legend(handles=handles, ncol=5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.03))
    fig.suptitle(f"Retail-only and wholesale-enabled value stacks ({title_suffix})", fontsize=12, fontweight="bold", y=1.08)
    _save_period_fig(fig, out_dir, "fig_financial_case_value_stack_subplots.png")
    plt.close(fig)

    # 5) Daily profit decomposition: WM/NWM x EV/BESS x energy/capacity.
    missing_decomp = [col for col in DECOMP_COMPONENTS if col not in period_df.columns]
    if missing_decomp:
        print(f"Skipping profit decomposition for {label}; rerun both MPC notebooks to create: {missing_decomp}")
    else:
        both_decomp = period_df[period_df['Case'] == 'Both'][['Scenario', 'Date', 'Total Revenue'] + DECOMP_COMPONENTS].copy()
        both_decomp = both_decomp.sort_values(['Scenario', 'Date'])
        both_decomp['Decomposition Residual'] = both_decomp['Total Revenue'] - both_decomp[DECOMP_COMPONENTS].sum(axis=1)
        both_decomp.to_csv(out_dir / 'profit_decomposition_daily.csv', index=False)
        decomp_total = both_decomp.groupby('Scenario', as_index=False)[['Total Revenue'] + DECOMP_COMPONENTS + ['Decomposition Residual']].sum(numeric_only=True)
        decomp_total = decomp_total.set_index('Scenario').reindex(SCENARIO_ORDER).dropna(how='all').reset_index()
        decomp_total.to_csv(out_dir / 'profit_decomposition_period_total.csv', index=False)

        group_handles = [Patch(facecolor=color, edgecolor='none', label=group) for group, color in DECOMP_GROUP_COLORS.items()]
        type_handles = [
            Patch(facecolor='#888888', edgecolor='#333333', label='Energy'),
            Patch(facecolor='#888888', edgecolor='#333333', hatch='////', label='Capacity'),
        ]
        decomp_handles = group_handles + type_handles
        decomp_ncol = max(2, min(len(decomp_handles), int(10.2 // 2.0)))

        fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.4), sharey=True, constrained_layout=False)
        fig.subplots_adjust(top=0.78, hspace=0.30, wspace=0.08)
        axes = axes.flatten()
        for ax, scenario in zip(axes, SCENARIO_ORDER):
            sub = both_decomp[both_decomp['Scenario'] == scenario].sort_values('Date')
            if sub.empty:
                ax.set_visible(False)
                continue
            x = np.arange(len(sub))
            pos_bottom = np.zeros(len(sub))
            neg_bottom = np.zeros(len(sub))
            for component in DECOMP_COMPONENTS:
                values = sub[component].fillna(0).to_numpy()
                bottoms = np.where(values >= 0, pos_bottom, neg_bottom)
                group = component.rsplit(' ', 1)[0]
                hatch = '////' if component.endswith('Capacity') else None
                ax.bar(x, values, bottom=bottoms, width=0.76, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.35)
                pos_bottom += np.where(values >= 0, values, 0)
                neg_bottom += np.where(values < 0, values, 0)
            ax.plot(x, sub['Total Revenue'], color='#111111', marker='o', markersize=2.8, linewidth=0.9, label='Net total')
            ax.axhline(0, color='#333333', linewidth=0.8)
            tick_step = max(1, int(np.ceil(len(sub) / 10)))
            ax.set_xticks(x[::tick_step])
            ax.set_xticklabels([pd.Timestamp(d).strftime('%m/%d') for d in sub['Date'].iloc[::tick_step]])
            ax.set_title(scenario.replace('-', ' + '), fontweight='bold', pad=5)
            ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
            ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
        axes[0].set_ylabel('Daily profit contribution (US$)')
        axes[2].set_ylabel('Daily profit contribution (US$)')
        fig.legend(handles=decomp_handles, ncol=decomp_ncol, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 0.925))
        fig.suptitle(f'Daily profit decomposition ({title_suffix})', fontsize=12, fontweight='bold', y=0.985)
        _save_period_fig(fig, out_dir, 'fig_profit_decomposition_daily.png')
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(8.5, 4.6), constrained_layout=True)
        x = np.arange(len(decomp_total))
        pos_bottom = np.zeros(len(decomp_total))
        neg_bottom = np.zeros(len(decomp_total))
        for component in DECOMP_COMPONENTS:
            values = decomp_total[component].fillna(0).to_numpy()
            bottoms = np.where(values >= 0, pos_bottom, neg_bottom)
            group = component.rsplit(' ', 1)[0]
            hatch = '////' if component.endswith('Capacity') else None
            ax.bar(x, values, bottom=bottoms, width=0.62, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.45)
            pos_bottom += np.where(values >= 0, values, 0)
            neg_bottom += np.where(values < 0, values, 0)
        net_values = decomp_total['Total Revenue'].to_numpy()
        ax.plot(x, net_values, color='#111111', marker='D', linestyle='none', markersize=4.6, label='Net total')
        for xi, yi in zip(x, net_values):
            ax.annotate(f'${yi:,.0f}', (xi, yi), textcoords='offset points', xytext=(0, 6 if yi >= 0 else -12), ha='center', fontsize=8)
        ax.axhline(0, color='#333333', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels([s.replace('-', '\n') for s in decomp_total['Scenario']])
        ax.set_ylabel('Period profit contribution (US$)')
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
        ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
        ax.legend(handles=decomp_handles, ncol=max(2, min(len(decomp_handles), int(8.5 // 2.0))), frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.16))
        ax.set_title(f'Period-total profit decomposition ({title_suffix})', fontweight='bold', pad=36)
        _save_period_fig(fig, out_dir, 'fig_profit_decomposition_period_total.png')
        plt.close(fig)

    return period_summary, period_delta


period_specs = []
if not all_df.empty:
    months = sorted(all_df["Date"].dt.to_period("M").unique())
    if len(months) > 1:
        full_label = "combined_" + _period_label(all_df)
        full_title = f"{pd.Timestamp(all_df['Date'].min()).strftime('%b %d')}--{pd.Timestamp(all_df['Date'].max()).strftime('%b %d, %Y')}"
        period_specs.append((full_label, all_df.copy(), full_title))

    for month in months:
        month_df = all_df[all_df["Date"].dt.to_period("M") == month].copy()
        period_specs.append((str(month), month_df, pd.Period(month).strftime("%B %Y")))

period_outputs = []
for label, period_df, title_suffix in period_specs:
    print(f"\nGenerating period figures: {label} ({len(period_df)} case-day rows)")
    period_summary, period_delta = plot_period_financials(period_df, label, title_suffix)
    period_outputs.append((label, len(period_df)))

print("\nGenerated period outputs:")
for label, n in period_outputs:
    print(f" - {label}: {n} case-day rows")




Generating period figures: 2025-06 (240 case-day rows)
Paper_Figures/financial_comparison/2025-06/fig_financial_violin_daily_total.png


Paper_Figures/financial_comparison/2025-06/fig_financial_delta_heatmap.png


Paper_Figures/financial_comparison/2025-06/fig_financial_case_comparison_subplots.png
Paper_Figures/financial_comparison/2025-06/fig_financial_case_value_stack_subplots.png


Paper_Figures/financial_comparison/2025-06/fig_profit_decomposition_daily.png
Paper_Figures/financial_comparison/2025-06/fig_profit_decomposition_period_total.png

Generated period outputs:
 - 2025-06: 240 case-day rows


## SEGAN Version 2.7: formal June 2025 paper figures

This section is independent of the legacy retail-only/Both 2 x 2 figures above. It reads the versioned formal Rolling/oracle snapshot and the latest complete Shrinking outputs, then writes journal-facing review figures to `Paper_Figures`. Only after validation should these files be copied to `Latex/paper2.7`.


In [4]:
# Formal June 2025 exports for the SEGAN v2.7 manuscript.
# Keep this code in the source plotting notebook; do not create a standalone plotting script.
FORMAL_REVIEW_DIR = PROJECT / 'Paper_Figures' / 'financial_comparison' / '2025-06'
FORMAL_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

FORMAL_ROOT = PROJECT / 'Validation_Results/June_2025/formal_runs/gate_e_20260726_c8250f11'
FORMAL_DAILY_SOURCES = [
    ('Shrinking 24 h Perfect', PROJECT / 'Results_Shrinking/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv'),
    ('Shrinking 24 h Persistence', PROJECT / 'Results_Shrinking/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv'),
    ('Rolling 24 h Perfect', FORMAL_ROOT / 'rolling/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv'),
    ('Rolling 24 h Persistence', FORMAL_ROOT / 'rolling/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv'),
]
ORACLE_DAILY_PATH = FORMAL_ROOT / 'rolling_perfect/oracle/oracle_daily_financial.csv'

required_paths = [path for _, path in FORMAL_DAILY_SOURCES] + [ORACLE_DAILY_PATH]
missing_formal = [path for path in required_paths if not path.exists()]
if missing_formal:
    raise FileNotFoundError('Missing formal paper inputs:\n' + '\n'.join(str(path) for path in missing_formal))

formal_frames = []
decomp_frames = []
for method, path in FORMAL_DAILY_SOURCES:
    frame = pd.read_csv(path)
    frame['Date'] = pd.to_datetime(frame['Date'])
    frame = frame[frame['Case'].eq('Both')].copy()
    if len(frame) != 30 or frame['Date'].nunique() != 30:
        raise ValueError(f'{method}: expected 30 unique June days, found {len(frame)} rows and {frame["Date"].nunique()} dates')
    frame['Method'] = method
    formal_frames.append(frame)
    decomp_frames.append(frame[['Method', 'Date', 'Total Revenue'] + DECOMP_COMPONENTS].copy())

oracle_daily = pd.read_csv(ORACLE_DAILY_PATH)
oracle_daily['Date'] = pd.to_datetime(oracle_daily['Date'])
if len(oracle_daily) != 30 or oracle_daily['Date'].nunique() != 30:
    raise ValueError('Oracle daily file does not contain exactly 30 unique June days')
oracle_daily['Method'] = 'Full billing-period Perfect oracle'
formal_frames.append(oracle_daily)

formal_daily = pd.concat(formal_frames, ignore_index=True, sort=False)
formal_daily = formal_daily.sort_values(['Method', 'Date'])
formal_daily.to_csv(FORMAL_REVIEW_DIR / 'formal_daily_financial.csv', index=False)

paper_method_order = [
    'Shrinking 24 h Perfect',
    'Shrinking 24 h Persistence',
    'Rolling 24 h Perfect',
    'Rolling 24 h Persistence',
    'Full billing-period Perfect oracle',
]
paper_method_labels = {
    'Shrinking 24 h Perfect': 'Shrinking\nPerfect',
    'Shrinking 24 h Persistence': 'Shrinking\nPersistence',
    'Rolling 24 h Perfect': 'Rolling\nPerfect',
    'Rolling 24 h Persistence': 'Rolling\nPersistence',
    'Full billing-period Perfect oracle': 'Full-period\nPerfect oracle',
}
paper_components = ['EV Revenue', 'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost']
formal_monthly = formal_daily.groupby('Method', as_index=False)[METRICS].sum(numeric_only=True)
formal_monthly = formal_monthly.set_index('Method').reindex(paper_method_order).reset_index()
formal_monthly.to_csv(FORMAL_REVIEW_DIR / 'formal_monthly_financial.csv', index=False)

def save_formal_review(fig, stem):
    fig.savefig(FORMAL_REVIEW_DIR / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)

# Figure 1: complete monthly net-value accounting for the five formal methods.
fig, ax = plt.subplots(figsize=(8.7, 4.7), constrained_layout=True)
y = np.arange(len(formal_monthly))
pos_left = np.zeros(len(formal_monthly))
neg_left = np.zeros(len(formal_monthly))
for component in paper_components:
    values = formal_monthly[component].to_numpy(float)
    left = np.where(values >= 0, pos_left, neg_left)
    ax.barh(y, values, left=left, height=0.60, color=COMPONENT_COLORS[component], edgecolor='white', linewidth=0.55, label=component)
    pos_left += np.where(values >= 0, values, 0)
    neg_left += np.where(values < 0, values, 0)
net_values = formal_monthly['Total Revenue'].to_numpy(float)
ax.scatter(net_values, y, color='#111111', marker='D', s=26, zorder=4, label='Net value')
for yi, value in zip(y, net_values):
    ax.annotate(f'${value:,.0f}', (value, yi), xytext=(6, 0), textcoords='offset points', va='center', fontsize=8)
ax.axvline(0, color='#333333', linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels([paper_method_labels[m] for m in formal_monthly['Method']])
ax.invert_yaxis()
ax.set_xlabel('June 2025 value (US$)')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis='x', color='#bdbdbd', alpha=0.32, linewidth=0.7)
ax.legend(ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.19))
ax.set_title('Executed monthly net value and financial components', fontweight='bold', pad=42)
save_formal_review(fig, 'fig_01_monthly_five_method_value_stack')

# Figure 2: daily and cumulative value for the controller emphasized in the paper.
focus_methods = ['Rolling 24 h Persistence', 'Rolling 24 h Perfect', 'Full billing-period Perfect oracle']
focus_colors = {
    'Rolling 24 h Persistence': '#E69F00',
    'Rolling 24 h Perfect': '#0072B2',
    'Full billing-period Perfect oracle': '#009E73',
}
fig, axes = plt.subplots(2, 1, figsize=(8.7, 5.7), sharex=True, constrained_layout=True)
for method in focus_methods:
    sub = formal_daily[formal_daily['Method'].eq(method)].sort_values('Date')
    axes[0].plot(sub['Date'], sub['Total Revenue'], color=focus_colors[method], linewidth=1.25, marker='o', markersize=2.7, label=method)
    axes[1].plot(sub['Date'], sub['Total Revenue'].cumsum(), color=focus_colors[method], linewidth=1.6, label=method)
axes[0].axhline(0, color='#333333', linewidth=0.8)
axes[0].set_ylabel('Daily net value (US$)')
axes[1].set_ylabel('Cumulative net value (US$)')
axes[1].set_xlabel('Date in June 2025')
for ax in axes:
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(color='#bdbdbd', alpha=0.30, linewidth=0.7)
axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[1].set_xlim(formal_daily['Date'].min(), formal_daily['Date'].max())
axes[0].legend(ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.27))
axes[0].set_title('Daily and cumulative executed value', fontweight='bold', pad=39)
save_formal_review(fig, 'fig_02_daily_rolling_oracle_value')

# Figure 3: four-policy WM/NWM, EV/BESS, energy/capacity attribution.
formal_decomp = pd.concat(decomp_frames, ignore_index=True)
formal_decomp.to_csv(FORMAL_REVIEW_DIR / 'formal_daily_profit_decomposition.csv', index=False)
decomp_total = formal_decomp.groupby('Method', as_index=False)[['Total Revenue'] + DECOMP_COMPONENTS].sum(numeric_only=True)
decomp_total = decomp_total.set_index('Method').reindex(paper_method_order[:-1]).reset_index()
decomp_total['Decomposition Residual'] = decomp_total['Total Revenue'] - decomp_total[DECOMP_COMPONENTS].sum(axis=1)
decomp_total.to_csv(FORMAL_REVIEW_DIR / 'formal_monthly_profit_decomposition.csv', index=False)
fig, ax = plt.subplots(figsize=(8.7, 4.8), constrained_layout=True)
x = np.arange(len(decomp_total))
pos_bottom = np.zeros(len(decomp_total))
neg_bottom = np.zeros(len(decomp_total))
for component in DECOMP_COMPONENTS:
    values = decomp_total[component].to_numpy(float)
    bottom = np.where(values >= 0, pos_bottom, neg_bottom)
    group = component.rsplit(' ', 1)[0]
    hatch = '////' if component.endswith('Capacity') else None
    ax.bar(x, values, bottom=bottom, width=0.62, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.45)
    pos_bottom += np.where(values >= 0, values, 0)
    neg_bottom += np.where(values < 0, values, 0)
ax.scatter(x, decomp_total['Total Revenue'], color='#111111', marker='D', s=28, zorder=4)
ax.axhline(0, color='#333333', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([paper_method_labels[m] for m in decomp_total['Method']])
ax.set_ylabel('June 2025 profit contribution (US$)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
legend_handles = [Patch(facecolor=color, edgecolor='none', label=group) for group, color in DECOMP_GROUP_COLORS.items()]
legend_handles += [Patch(facecolor='#888888', edgecolor='#333333', label='Energy'), Patch(facecolor='#888888', edgecolor='#333333', hatch='////', label='Capacity')]
ax.legend(handles=legend_handles, ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.22))
ax.set_title('Executed profit attribution by resource and service channel', fontweight='bold', pad=82)
save_formal_review(fig, 'fig_03_monthly_resource_service_decomposition')

# Figure 4: daily attribution retained as a paper appendix/supplement candidate.
fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.4), sharey=True, constrained_layout=False)
fig.subplots_adjust(top=0.80, hspace=0.30, wspace=0.10)
for ax, method in zip(axes.flat, paper_method_order[:-1]):
    sub = formal_decomp[formal_decomp['Method'].eq(method)].sort_values('Date')
    xday = np.arange(len(sub))
    pos_bottom = np.zeros(len(sub))
    neg_bottom = np.zeros(len(sub))
    for component in DECOMP_COMPONENTS:
        values = sub[component].to_numpy(float)
        bottom = np.where(values >= 0, pos_bottom, neg_bottom)
        group = component.rsplit(' ', 1)[0]
        hatch = '////' if component.endswith('Capacity') else None
        ax.bar(xday, values, bottom=bottom, width=0.78, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.25)
        pos_bottom += np.where(values >= 0, values, 0)
        neg_bottom += np.where(values < 0, values, 0)
    ax.plot(xday, sub['Total Revenue'], color='#111111', linewidth=0.9, marker='o', markersize=2.4)
    ax.axhline(0, color='#333333', linewidth=0.8)
    ax.set_title(method.replace(' 24 h ', ' + '), fontweight='bold')
    ax.set_xticks(xday[::4])
    ax.set_xticklabels([pd.Timestamp(d).strftime('%m/%d') for d in sub['Date'].iloc[::4]])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
axes[0, 0].set_ylabel('Daily profit contribution (US$)')
axes[1, 0].set_ylabel('Daily profit contribution (US$)')
fig.legend(handles=legend_handles, ncol=6, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 0.92))
fig.suptitle('Daily executed profit attribution', fontsize=12, fontweight='bold', y=0.985)
save_formal_review(fig, 'fig_04_daily_resource_service_decomposition')

print('SEGAN v2.7 formal review figures written to:', FORMAL_REVIEW_DIR.relative_to(PROJECT))
print(formal_monthly[['Method', 'Total Revenue']].to_string(index=False))


SEGAN v2.7 formal review figures written to: Paper_Figures/financial_comparison/2025-06
                            Method  Total Revenue
            Shrinking 24 h Perfect   10552.670000
        Shrinking 24 h Persistence   10985.420000
              Rolling 24 h Perfect   10733.150000
          Rolling 24 h Persistence   10485.280000
Full billing-period Perfect oracle   12565.154573
